# 12 — RDS PostgreSQL → Databricks Unity Catalog Pipeline

> **Goal**: Demonstrate the end-to-end RDS decommission migration flow:  
> PostgreSQL RDS → Databricks (Databricks Connect) → Unity Catalog

This notebook covers:
1. Connecting to Databricks serverless via Databricks Connect
2. Reading tables from PostgreSQL RDS via JDBC
3. Writing to Unity Catalog as managed Delta tables (full load)
4. Running incremental CDC updates using watermarks
5. Running the reporting transformations (migrated from PostgreSQL stored procedures)

## Architecture

```
PostgreSQL RDS  →  JDBC  →  Databricks Spark  →  Unity Catalog (Delta)
(source)                    (Databricks Connect      sparkling.{schema}.*
                             or Databricks Job)
```

## Deployment options

| Option | Use when | Command |
|--------|----------|---------|
| This notebook | Interactive exploration / one-off runs | Run cells below |
| Databricks Asset Bundle | Scheduled production runs | `databricks bundle run` |

The same Python logic lives in `pipelines/rds_data_migration_1/` as a deployable bundle.

## 0 — Configuration

In [ ]:
# ── Databricks Connect session ────────────────────────────────────────────
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.serverless().getOrCreate()
print(f"✅ Connected — Spark {spark.version}")

In [ ]:
import os
from datetime import date, datetime
from pyspark.sql import functions as F

# ── Unity Catalog target ──────────────────────────────────────────────────
CATALOG = "sparkling"
SCHEMA  = "banking"          # change to your username in dev

# ── RDS connection (use Databricks secrets in production) ─────────────────
# In production these come from {{secrets/sparkling/rds_*}} in the job YAML.
# For local notebook use, either set environment variables or paste values
# directly (never commit real credentials).
RDS_HOST     = os.getenv("RDS_HOST", "sparkling-postgres-db.c3oweiuwo3lj.ap-southeast-1.rds.amazonaws.com")
RDS_PORT     = os.getenv("RDS_PORT", "5432")
RDS_DATABASE = os.getenv("RDS_DATABASE", "sparkdb")
RDS_USERNAME = os.getenv("RDS_USERNAME", "sparkadmin")
RDS_PASSWORD = os.getenv("RDS_PASSWORD", "")  # set via env var

JDBC_URL  = f"jdbc:postgresql://{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"
JDBC_PROPS = {"user": RDS_USERNAME, "password": RDS_PASSWORD, "driver": "org.postgresql.Driver"}

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"RDS URL : {JDBC_URL}")

## 1 — Full Load: Read all RDS tables and write to Unity Catalog

In [ ]:
# Ensure target schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema {CATALOG}.{SCHEMA} ready")

In [ ]:
# Tables to migrate (all RDS banking tables)
TABLES = [
    "dim_date",
    "dim_branch",
    "dim_account_type",
    "dim_customer",
    "dim_account",
    "fact_transaction",
    "fact_daily_balance",
]

for table_name in TABLES:
    target = f"{CATALOG}.{SCHEMA}.{table_name}"
    print(f"Reading {table_name} from RDS...")
    df = spark.read.jdbc(url=JDBC_URL, table=table_name, properties=JDBC_PROPS)
    count = df.count()
    print(f"  {count:>10,} rows → writing to {target}")
    df.write.format("delta").mode("overwrite").saveAsTable(target)
    print(f"  ✅ Done")

print("\n🎉 Full load complete!")

## 2 — Validate: Row counts in Unity Catalog vs RDS

In [ ]:
print(f"{'Table':<30} {'UC rows':>12} {'RDS rows':>12} {'Match?':>8}")
print("-" * 70)

for table_name in TABLES:
    uc_count  = spark.table(f"{CATALOG}.{SCHEMA}.{table_name}").count()
    rds_df    = spark.read.jdbc(url=JDBC_URL, table=table_name, properties=JDBC_PROPS)
    rds_count = rds_df.count()
    match     = "✅" if uc_count == rds_count else "❌"
    print(f"{table_name:<30} {uc_count:>12,} {rds_count:>12,} {match:>8}")

## 3 — Incremental CDC: Update Unity Catalog with new/changed RDS rows

In [ ]:
# This cell mirrors the logic in rds_incremental.py
# CDC watermarks are stored in the sparkling.banking.cdc_watermark Delta table.

# Ensure watermark table exists
WM_TABLE = f"{CATALOG}.{SCHEMA}.cdc_watermark"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WM_TABLE} (
        table_name      STRING NOT NULL,
        last_watermark  TIMESTAMP,
        last_row_count  BIGINT,
        last_run_status STRING,
        updated_at      TIMESTAMP
    ) USING DELTA
""")

def get_watermark(table_name):
    rows = spark.sql(f"SELECT last_watermark FROM {WM_TABLE} WHERE table_name = '{table_name}'").collect()
    if rows and rows[0]["last_watermark"]:
        return rows[0]["last_watermark"].strftime("%Y-%m-%d %H:%M:%S")
    return "2020-01-01 00:00:00"

print("Current watermarks:")
spark.table(WM_TABLE).show(truncate=False)

In [ ]:
# Incremental load for dim_customer (SCD2 — append new versions)
wm = get_watermark("dim_customer")
print(f"dim_customer watermark: {wm}")

query = f"(SELECT * FROM dim_customer WHERE last_modified > '{wm}') AS incremental"
df_new = spark.read.jdbc(url=JDBC_URL, table=query, properties=JDBC_PROPS)
count = df_new.count()
print(f"New/changed rows: {count:,}")

if count > 0:
    df_new.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_customer")
    new_wm = df_new.agg(F.max("last_modified")).collect()[0][0]
    spark.sql(f"""
        MERGE INTO {WM_TABLE} AS t
        USING (SELECT 'dim_customer' AS table_name, TIMESTAMP('{new_wm}') AS last_watermark,
                      {count} AS last_row_count, 'SUCCESS' AS last_run_status, CURRENT_TIMESTAMP AS updated_at) AS s
        ON t.table_name = s.table_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Appended {count:,} rows; new watermark: {new_wm}")
else:
    print("✅ No new rows — watermark unchanged")

In [ ]:
# Incremental MERGE for dim_account (SCD1 — upsert by account_id)
wm = get_watermark("dim_account")
print(f"dim_account watermark: {wm}")

query = f"(SELECT * FROM dim_account WHERE last_modified > '{wm}') AS incremental"
df_new = spark.read.jdbc(url=JDBC_URL, table=query, properties=JDBC_PROPS)
count = df_new.count()
print(f"New/changed rows: {count:,}")

if count > 0:
    df_new.createOrReplaceTempView("dim_account_incremental")
    target = f"{CATALOG}.{SCHEMA}.dim_account"
    all_cols = df_new.columns
    update_set = ", ".join(f"t.{c} = s.{c}" for c in all_cols if c != "account_id")
    insert_cols = ", ".join(all_cols)
    insert_vals = ", ".join(f"s.{c}" for c in all_cols)
    spark.sql(f"""
        MERGE INTO {target} AS t
        USING dim_account_incremental AS s
        ON t.account_id = s.account_id
        WHEN MATCHED THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
    """)
    print(f"✅ Merged {count:,} rows into dim_account")
else:
    print("✅ No new rows")

## 4 — Reporting Transformations (Stored Procedure Migration)

In [ ]:
# Add project root to path so we can import from pipelines/
import sys, pathlib
proj_root = str(pathlib.Path().resolve().parent)
bundle_src = str(pathlib.Path(proj_root) / "pipelines" / "rds_data_migration_1" / "src")
if bundle_src not in sys.path:
    sys.path.insert(0, bundle_src)

from rds_data_migration_1_etl.transformations.rds_reporting_transform import run_daily_reporting

run_daily_reporting(spark, CATALOG, SCHEMA, run_date=date.today())
print("✅ All reporting tables updated")

## 5 — Query Reporting Tables

In [ ]:
# Monthly transaction summary — top 10 customers by net flow
spark.sql(f"""
    SELECT customer_id, segment, account_type_code,
           txn_count, net_flow_vnd, preferred_channel
    FROM {CATALOG}.{SCHEMA}.rpt_monthly_txn_summary
    ORDER BY ABS(net_flow_vnd) DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Customer segment KPIs
spark.sql(f"""
    SELECT segment, total_customers, active_accounts,
           ROUND(total_balance_vnd / 1e9, 2) AS total_balance_bn_vnd,
           dormant_account_pct
    FROM {CATALOG}.{SCHEMA}.rpt_customer_segment_kpi
    ORDER BY total_balance_vnd DESC
""").show()

In [ ]:
# Digital channel share
spark.sql(f"""
    SELECT channel, txn_count, pct_of_total_txns, digital_flag
    FROM {CATALOG}.{SCHEMA}.rpt_channel_analysis
    ORDER BY txn_count DESC
""").show()

In [ ]:
# Dormant watchlist — accounts at risk
spark.sql(f"""
    SELECT account_id, customer_name, segment, days_inactive,
           dormancy_risk, recommended_action
    FROM {CATALOG}.{SCHEMA}.rpt_dormant_watchlist
    ORDER BY days_inactive DESC
    LIMIT 20
""").show(truncate=False)

## 6 — Production Deployment via Databricks Asset Bundle

The Databricks Asset Bundle in `pipelines/rds_data_migration_1/` packages the same
Python code as a wheel and deploys it as scheduled Databricks Jobs.

### One-time setup

```bash
# Store RDS credentials in Databricks secret scope (never hardcode)
databricks secrets create-scope sparkling
databricks secrets put-secret sparkling rds_host     --string-value "<endpoint>"
databricks secrets put-secret sparkling rds_username --string-value "sparkadmin"
databricks secrets put-secret sparkling rds_password --string-value "<password>"
```

### Deploy and run

```bash
cd pipelines/rds_data_migration_1

# Deploy to production
databricks bundle deploy --target prod

# Run full initial load (once)
databricks bundle run rds_data_migration_1_job --target prod

# Run daily incremental + reporting (repeatable)
databricks bundle run rds_daily_job --target prod
```

The `rds_daily_job` is also scheduled to run automatically at 02:00 UTC every day.

### Job dependency graph

```
rds_data_migration_1_job  (run once)
  └── rds_ingest_task     Full load of all 7 tables

rds_daily_job             (daily at 02:00 UTC)
  ├── rds_incremental_task  CDC watermark-based incremental load
  └── rds_reporting_task    Reporting transformations (depends on incremental)
```

In [ ]:
spark.stop()
print("Session closed.")